In [ ]:
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

ROOT = Path("/kaggle/working/FastSpeech2_Submission")
ROOT.mkdir(parents=True, exist_ok=True)

repo = ROOT / "FastSpeech2"

if not (repo / "train.py").exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/ming024/FastSpeech2.git",
            str(repo)
        ],
        check=True
    )

os.chdir(repo)

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")
print("Repository:", repo)
print("Git commit:", subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    text=True
).strip())

print("\nKey dependencies:")
for name in ["librosa", "numpy", "scipy", "yaml", "g2p_en", "pyworld", "tgt", "phonemizer"]:
    print(name, "AVAILABLE" if importlib.util.find_spec(name) else "MISSING")

print("\nExisting Kaggle datasets:")
data_root = Path("/kaggle/input")
for p in sorted(data_root.iterdir()) if data_root.exists() else []:
    print(" -", p.name)

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("datasets") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "datasets"
    ])

from huggingface_hub import HfApi

api = HfApi()

datasets_to_check = {
    "English": "ai4bharat/Svarah",
    "Telugu": "SPRINGLab/IndicTTS_Telugu"
}

for lang, repo_id in datasets_to_check.items():
    try:
        info = api.dataset_info(repo_id)
        print(f"{lang}: ACCESSIBLE")
        print("Dataset:", repo_id)
        print("Private:", info.private)
        print("Files:", len(info.siblings))
    except Exception as e:
        print(f"{lang}: ERROR")
        print(type(e).__name__, str(e)[:300])

In [ ]:
from datasets import load_dataset

repos = {
    "English": ("ai4bharat/Svarah", "test"),
    "Telugu": ("SPRINGLab/IndicTTS_Telugu", "train")
}

for lang, (repo, split) in repos.items():
    print(f"\n{'='*50}")
    print(lang)

    ds = load_dataset(repo, split=split, streaming=True)

    print("Columns:", ds.column_names)

    sample = next(iter(ds))

    for key, value in sample.items():
        if key in ("audio", "audio_filepath"):
            print(key, "Audio field present")
        else:
            print(key, str(value)[:150])

In [ ]:
import os, csv, json, random
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
from datasets import load_dataset, Audio

ROOT = Path("/kaggle/working/FastSpeech2_Submission")
SR = 22050
SEED = 2026
TARGET = {"train": 5 * 3600, "val": 3600}

sources = {
    "English": ("ai4bharat/Svarah", "test", "audio_filepath"),
    "Telugu": ("SPRINGLab/IndicTTS_Telugu", "train", "audio")
}

summary = {}

for lang, (repo, split, audio_col) in sources.items():
    print(f"\nProcessing {lang}...", flush=True)

    ds = load_dataset(repo, split=split, streaming=True, token=True)
    ds = ds.cast_column(audio_col, Audio(decode=False))

    out = ROOT / lang
    for part in TARGET:
        (out / "wavs" / part).mkdir(parents=True, exist_ok=True)

    rows = {"train": [], "val": []}
    seconds = {"train": 0.0, "val": 0.0}
    rng = random.Random(SEED)
    count = 0

    for example in ds:
        if all(seconds[p] >= TARGET[p] for p in TARGET):
            break

        text = str(example.get("text", "")).strip()
        if not text:
            continue

        audio = example[audio_col]

        try:
            if audio.get("bytes") is not None:
                import io
                wav, sr = sf.read(io.BytesIO(audio["bytes"]))
            elif audio.get("path"):
                wav, sr = sf.read(audio["path"])
            else:
                continue

            if wav.ndim > 1:
                wav = wav.mean(axis=1)

            duration = len(wav) / sr
            if not 1.0 <= duration <= 30.0:
                continue

            wav = librosa.resample(
                wav.astype(np.float32),
                orig_sr=sr,
                target_sr=SR
            )

            available = [p for p in TARGET if seconds[p] < TARGET[p]]
            if not available:
                break

            part = rng.choice(available)
            filename = f"{lang.lower()}_{count:06d}.wav"
            path = out / "wavs" / part / filename

            sf.write(path, wav, SR)

            rows[part].append({
                "audio": str(path),
                "text": text,
                "duration": round(len(wav) / SR, 4)
            })

            seconds[part] += len(wav) / SR
            count += 1

            if count % 250 == 0:
                print(
                    f"{lang}: {count} files | "
                    f"train {seconds['train']/3600:.2f}h | "
                    f"val {seconds['val']/3600:.2f}h",
                    flush=True
                )

        except Exception as e:
            if count < 5:
                print("Skipped:", type(e).__name__, str(e)[:100])
            continue

    for part in TARGET:
        manifest = out / f"{part}_manifest.csv"
        with open(manifest, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["audio", "text", "duration"])
            writer.writeheader()
            writer.writerows(rows[part])

    summary[lang] = {
        part: {
            "samples": len(rows[part]),
            "hours": round(seconds[part] / 3600, 4)
        }
        for part in TARGET
    }

    print(lang, summary[lang], flush=True)

with open(ROOT / "data_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nDATA SUMMARY")
print(json.dumps(summary, indent=2))

In [ ]:
import csv
import inspect
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import ctc_forced_aligner as ctc

ROOT = Path("/kaggle/working/FastSpeech2_Submission")

print("Alignment package:", ctc.__file__)

print("\nModel and tokenizer API:")
for name in ["ensure_onnx_model", "Tokenizer", "preprocess_text"]:
    obj = getattr(ctc, name)
    print(name, inspect.signature(obj))

print("\nAlignment model constants:")
print("Model URL:", ctc.MODEL_URL)
print("Sampling frequency:", ctc.SAMPLING_FREQ)

print("\nSample recordings:")

for lang in ["English", "Telugu"]:
    manifest = ROOT / lang / "train_manifest.csv"

    with open(manifest, encoding="utf-8") as f:
        row = next(csv.DictReader(f))

    audio, sr = sf.read(row["audio"])

    print(f"\n{lang}")
    print("Text:", row["text"])
    print("Duration:", round(len(audio) / sr, 2), "seconds")
    print("Sample rate:", sr)
    print("Audio shape:", audio.shape)

In [ ]:
import csv
from pathlib import Path

import numpy as np
import ctc_forced_aligner as ctc

ROOT = Path("/kaggle/working/FastSpeech2_Submission")
MODEL_PATH = ROOT / "alignment_test" / "ctc_model.onnx"

assert MODEL_PATH.exists(), "CTC model is missing"

aligner = ctc.Alignment(str(MODEL_PATH))

results = {}

for lang, code in [("English", "eng"), ("Telugu", "tel")]:
    with open(ROOT / lang / "train_manifest.csv", encoding="utf-8") as f:
        row = next(csv.DictReader(f))

    print(f"\n{'=' * 50}")
    print("Language:", lang)
    print("Text:", row["text"])

    try:
        audio = ctc.load_audio(row["audio"])

        emissions, stride = ctc.generate_emissions(
            aligner.model,
            audio,
            batch_size=1
        )

        tokens, text_starred = ctc.preprocess_text(
            row["text"],
            romanize=True,
            language=code
        )

        segments, scores, blank = ctc.get_alignments(
            emissions,
            tokens,
            aligner.tokenizer
        )

        spans = ctc.get_spans(tokens, segments, blank)

        timestamps = ctc.postprocess_results(
            text_starred,
            spans,
            stride,
            scores
        )

        results[lang] = timestamps

        print("Emission shape:", np.shape(emissions))
        print("Number of aligned segments:", len(timestamps))

        for item in timestamps[:12]:
            print(item)

    except Exception as e:
        print("Alignment error:", type(e).__name__, str(e)[:500])
        raise

In [ ]:
import csv
from pathlib import Path
import ctc_forced_aligner as ctc

ROOT = Path("/kaggle/working/FastSpeech2_Submission")
MODEL_PATH = ROOT / "alignment_test" / "ctc_model.onnx"

aligner = ctc.Alignment(str(MODEL_PATH))

with open(ROOT / "Telugu" / "train_manifest.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

row = next(
    r for r in rows
    if 5 <= len(r["text"].split()) <= 15
    and 3 <= float(r["duration"]) <= 15
)

print("Text:", row["text"])
print("Audio duration:", row["duration"], "seconds")

audio = ctc.load_audio(row["audio"])

emissions, stride = ctc.generate_emissions(
    aligner.model, audio, batch_size=1
)

tokens, text_starred = ctc.preprocess_text(
    row["text"], romanize=True, language="tel"
)

segments, scores, blank = ctc.get_alignments(
    emissions, tokens, aligner.tokenizer
)

spans = ctc.get_spans(tokens, segments, blank)

timestamps = ctc.postprocess_results(
    text_starred, spans, stride, scores
)

print("\nAligned words:", len(timestamps))

for item in timestamps:
    print(item)

# Preprocessing

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/FastSpeech2_Submission")

if shutil.which("espeak-ng") is None:
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
        stdout=subprocess.DEVNULL
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "espeak-ng"],
        check=True,
        stdout=subprocess.DEVNULL
    )

frontend_code = r'''
import re
import unicodedata
from functools import lru_cache
from phonemizer.backend import EspeakBackend
from phonemizer.separator import Separator

LANGUAGES = {
    "English": "en-us",
    "Telugu": "te",
}

@lru_cache(maxsize=2)
def backend(language):
    return EspeakBackend(
        language=LANGUAGES[language],
        preserve_punctuation=False,
        with_stress=False,
        language_switch="remove-flags"
    )

def phonemize_words(text, language):
    text = unicodedata.normalize("NFC", text.strip())
    text = re.sub(r"\s+", " ", text)

    if not text:
        return []

    words = text.split(" ")
    result = []

    sep = Separator(phone="|", word=" ", syllable="")

    for word in words:
        output = backend(language).phonemize(
            [word],
            separator=sep,
            strip=True,
            njobs=1
        )[0]

        phones = [p for p in output.split("|") if p.strip()]
        phones = [p.strip() for p in phones]

        if phones:
            result.append(phones)

    return result

def phonemize_text(text, language):
    words = phonemize_words(text, language)
    return [phone for word in words for phone in word]
'''

frontend_path = ROOT / "phoneme_frontend.py"
frontend_path.write_text(frontend_code, encoding="utf-8")

sys.path.insert(0, str(ROOT))

from phoneme_frontend import phonemize_words, phonemize_text

examples = {
    "English": "The orchestra changed its rhythm unexpectedly.",
    "Telugu": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది."
}

for language, sentence in examples.items():
    print(f"\n{language}")
    print("Original:", sentence)

    try:
        words = phonemize_words(sentence, language)
        phones = phonemize_text(sentence, language)

        print("Phonemes by word:", words)
        print("Total phonemes:", len(phones))
        print("Flattened:", phones[:60])

        assert len(words) == len(sentence.split())
        assert len(phones) > 0

    except Exception as e:
        print("ERROR:", type(e).__name__, str(e)[:500])
        raise

print("\nFrontend saved:", frontend_path)

# Production acoustic preprocessing

In [ ]:
import csv
import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
import librosa
import pyworld as pw
import ctc_forced_aligner as ctc

from phoneme_frontend import phonemize_words

ROOT = Path("/kaggle/working/FastSpeech2_Submission")
MODEL = ROOT / "alignment_test" / "ctc_model.onnx"
OUT = ROOT / "acoustic_features"

SR = 22050
HOP = 256
FFT = 1024
MELS = 80

assert MODEL.exists(), "Previously downloaded CTC model is missing"

aligner = ctc.Alignment(str(MODEL))

def word_to_phone_durations(timestamps, words, frames, seconds):
    if len(timestamps) != len(words):
        raise ValueError(
            f"Word count mismatch: {len(timestamps)} vs {len(words)}"
        )

    if not words or frames <= 0:
        raise ValueError("Empty phonemes or mel frames")

    boundaries = [0.0]

    for a, b in zip(timestamps[:-1], timestamps[1:]):
        midpoint = (
            float(a["end"]) + float(b["start"])
        ) / 2
        boundaries.append(midpoint)

    boundaries.append(seconds)

    boundaries = np.clip(boundaries, 0, seconds)
    boundaries = np.maximum.accumulate(boundaries)

    frame_boundaries = np.rint(
        np.asarray(boundaries) / seconds * frames
    ).astype(int)

    frame_boundaries[0] = 0
    frame_boundaries[-1] = frames

    durations = []

    for i, phones in enumerate(words):
        count = frame_boundaries[i + 1] - frame_boundaries[i]
        n = len(phones)

        if count < n:
            raise ValueError("Insufficient frames for phonemes")

        q, r = divmod(count, n)
        durations.extend(
            [q + 1] * r + [q] * (n - r)
        )

    durations = np.asarray(durations, dtype=np.int32)

    if durations.sum() != frames:
        raise ValueError("Duration and mel lengths differ")

    return durations


def acoustic_features(wav):
    mel = librosa.feature.melspectrogram(
        y=wav,
        sr=SR,
        n_fft=FFT,
        hop_length=HOP,
        win_length=FFT,
        n_mels=MELS,
        fmin=0,
        fmax=8000,
        center=True,
        power=2.0
    )

    mel = np.log(np.maximum(mel, 1e-8)).T.astype(np.float32)
    frames = len(mel)

    stft = librosa.stft(
        wav,
        n_fft=FFT,
        hop_length=HOP,
        center=True
    )

    energy = np.sqrt(
        np.sum(np.abs(stft) ** 2, axis=0)
    ).astype(np.float32)

    audio64 = np.ascontiguousarray(wav, dtype=np.float64)

    f0, times = pw.dio(
        audio64,
        SR,
        frame_period=HOP / SR * 1000
    )

    f0 = pw.stonemask(audio64, f0, times, SR)

    if len(f0) != frames:
        f0 = np.interp(
            np.linspace(0, 1, frames),
            np.linspace(0, 1, len(f0)),
            f0
        )

    if len(energy) != frames:
        energy = np.interp(
            np.linspace(0, 1, frames),
            np.linspace(0, 1, len(energy)),
            energy
        )

    return (
        mel,
        np.asarray(f0, dtype=np.float32),
        np.asarray(energy, dtype=np.float32)
    )


def process_sample(row, lang, code, output):
    wav, sr = librosa.load(
        row["audio"],
        sr=SR,
        mono=True
    )

    text = row["text"].strip()
    words = phonemize_words(text, lang)
    phones = [p for word in words for p in word]

    if not phones:
        raise ValueError("No phonemes")

    audio16 = ctc.load_audio(row["audio"])

    emissions, stride = ctc.generate_emissions(
        aligner.model,
        audio16,
        batch_size=1
    )

    tokens, starred = ctc.preprocess_text(
        text,
        romanize=True,
        language=code
    )

    segments, scores, blank = ctc.get_alignments(
        emissions,
        tokens,
        aligner.tokenizer
    )

    spans = ctc.get_spans(tokens, segments, blank)

    timestamps = ctc.postprocess_results(
        starred,
        spans,
        stride,
        scores
    )

    mel, pitch, energy = acoustic_features(wav)

    duration = word_to_phone_durations(
        timestamps,
        words,
        len(mel),
        len(wav) / SR
    )

    if len(duration) != len(phones):
        raise ValueError("Phoneme-duration length mismatch")

    if not np.isfinite(mel).all():
        raise ValueError("Invalid mel values")

    if not np.isfinite(pitch).all():
        raise ValueError("Invalid pitch values")

    if not np.isfinite(energy).all():
        raise ValueError("Invalid energy values")

    np.savez(
        output.with_suffix(".npz"),
        mel=mel,
        pitch=pitch,
        energy=energy,
        duration=duration
    )

    metadata = {
        "audio": row["audio"],
        "text": text,
        "phones": phones,
        "frames": len(mel),
        "duration_seconds": len(wav) / SR,
        "alignment_type": "CTC word-level; estimated within-word phoneme durations"
    }

    output.with_suffix(".json").write_text(
        json.dumps(metadata, ensure_ascii=False),
        encoding="utf-8"
    )

    return len(phones), len(mel)


start = time.time()

for lang, code in [("English", "eng"), ("Telugu", "tel")]:
    print(f"\nProcessing {lang}", flush=True)

    for split in ["train", "val"]:
        folder = OUT / lang / split
        folder.mkdir(parents=True, exist_ok=True)

        manifest = ROOT / lang / f"{split}_manifest.csv"

        with open(manifest, encoding="utf-8") as f:
            rows = list(csv.DictReader(f))

        completed = 0
        failed = 0

        for i, row in enumerate(rows):
            stem = Path(row["audio"]).stem
            target = folder / stem

            if (
                target.with_suffix(".npz").exists()
                and target.with_suffix(".json").exists()
            ):
                completed += 1
                continue

            try:
                nphones, nframes = process_sample(
                    row, lang, code, target
                )

                completed += 1

                if completed <= 3:
                    print(
                        stem,
                        "phonemes:", nphones,
                        "frames:", nframes,
                        flush=True
                    )

            except Exception as e:
                failed += 1

                if failed <= 5:
                    print(
                        "Skipped", stem,
                        type(e).__name__,
                        str(e)[:160],
                        flush=True
                    )

            if (i + 1) % 25 == 0:
                print(
                    f"{lang}/{split}: "
                    f"{i+1}/{len(rows)} examined, "
                    f"{completed} saved, "
                    f"{failed} skipped, "
                    f"{(time.time()-start)/60:.1f} min",
                    flush=True
                )

        print(
            f"{lang}/{split}: {completed} saved, "
            f"{failed} skipped",
            flush=True
        )

    counter = Counter()

    for meta_file in (OUT / lang).rglob("*.json"):
        meta = json.loads(meta_file.read_text(encoding="utf-8"))
        counter.update(meta["phones"])

    vocab = {
        "<pad>": 0,
        "<unk>": 1
    }

    for phone in sorted(counter):
        if phone not in vocab:
            vocab[phone] = len(vocab)

    (OUT / lang / "vocab.json").write_text(
        json.dumps(vocab, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(f"{lang} vocabulary: {len(vocab)} tokens", flush=True)

print(
    "\nFeature extraction finished in",
    round((time.time() - start) / 60, 1),
    "minutes"
)

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/FastSpeech2_Submission")

for lang in ["English", "Telugu"]:
    print(f"\n{'='*15} {lang} {'='*15}")

    paths = {
        "Training features": ROOT / "acoustic_features" / lang / "train",
        "Validation features": ROOT / "acoustic_features" / lang / "val",
        "Vocabulary": ROOT / "acoustic_features" / lang / "vocab.json",
        "Best checkpoint": ROOT / lang / "checkpoints" / "best.pt",
        "Last checkpoint": ROOT / lang / "checkpoints" / "last.pt",
        "Normalized checkpoint": ROOT / lang / "checkpoints" / "normalized_best.pt",
        "Training history": ROOT / "Results" / lang / "history.json",
        "FastSpeech2 audio": ROOT / "Results" / lang / "test_generated_audios",
        "Working VITS audio": ROOT / "Results" / lang / "pretrained_baseline",
    }

    for name, path in paths.items():
        if path.is_dir():
            count = len(list(path.glob("*.npz"))) if "features" in name.lower() else len(list(path.glob("*.wav")))
            print(f"{name}: EXISTS ({count} files)")
        else:
            print(f"{name}: {'EXISTS' if path.exists() else 'MISSING'}")

print("\nSaved notebooks:")
for p in ROOT.rglob("*.ipynb"):
    print(p)

In [ ]:
import json
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader

ROOT = Path("/kaggle/working/FastSpeech2_Submission")
FEATURES = ROOT / "acoustic_features"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class TTSDataset(Dataset):
    def __init__(self, lang, split):
        self.folder = FEATURES / lang / split

        with open(FEATURES / lang / "vocab.json", encoding="utf-8") as f:
            self.vocab = json.load(f)

        self.files = sorted(self.folder.glob("*.npz"))

        self.files = [
            p for p in self.files
            if p.with_suffix(".json").exists()
        ]

        print(f"{lang}/{split}: {len(self.files)} samples")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]

        with np.load(path) as data:
            mel = data["mel"].copy()
            pitch = data["pitch"].copy()
            energy = data["energy"].copy()
            duration = data["duration"].copy()

        meta = json.loads(
            path.with_suffix(".json").read_text(encoding="utf-8")
        )

        phones = [
            self.vocab.get(p, 1)
            for p in meta["phones"]
        ]

        return {
            "phones": torch.tensor(phones, dtype=torch.long),
            "duration": torch.tensor(duration, dtype=torch.long),
            "mel": torch.tensor(mel, dtype=torch.float32),
            "pitch": torch.tensor(pitch, dtype=torch.float32),
            "energy": torch.tensor(energy, dtype=torch.float32),
            "text": meta["text"]
        }


def collate(batch):
    from torch.nn.utils.rnn import pad_sequence

    result = {}

    for key in ["phones", "duration", "mel", "pitch", "energy"]:
        result[key] = pad_sequence(
            [item[key] for item in batch],
            batch_first=True,
            padding_value=0
        )

    result["phone_lengths"] = torch.tensor(
        [len(item["phones"]) for item in batch]
    )

    result["mel_lengths"] = torch.tensor(
        [len(item["mel"]) for item in batch]
    )

    result["text"] = [item["text"] for item in batch]

    return result


for lang in ["English", "Telugu"]:
    ds = TTSDataset(lang, "train")

    if len(ds):
        sample = ds[0]
        print(
            lang,
            "Phones:", sample["phones"].shape,
            "Mel:", sample["mel"].shape,
            "Duration sum:", sample["duration"].sum().item()
        )

In [ ]:
import torch
import numpy as np
import soundfile as sf

from pathlib import Path
from transformers import VitsTokenizer, VitsModel
from IPython.display import Audio, display

ROOT = Path("/kaggle/working/FastSpeech2_Submission")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SENTENCES = {
    "English": [
        "The antidisestablishmentarianism movement gained considerable momentum during the parliamentary proceedings.",
        "The orchestra's conduct during the ceremony was praised by the audience.",
        "The researchers will conduct an experiment to evaluate the proposed method.",
        "The rhythm of the orchestra changed unexpectedly.",
        "The meeting will start at ten forty five in the morning on November twelfth."
    ],
    "Telugu": [
        "నూతన ఔషధాల ప్రభావాన్ని పరిశోధకులు విశ్లేషించారు.",
        "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
        "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
        "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
        "ఈ సమావేశం నవంబర్ పన్నెండవ తేదీన ఉదయం ప్రారంభమవుతుంది."
    ]
}

for lang in ["English", "Telugu"]:
    local = ROOT / "pretrained_models" / lang
    output = ROOT / "Results" / lang / "pretrained_baseline"
    output.mkdir(parents=True, exist_ok=True)

    existing = list(output.glob("test_*.wav"))

    if len(existing) == 5:
        print(f"{lang}: All 5 WAV files already exist")
        display(Audio(filename=str(output / "test_02.wav")))
        continue

    tokenizer = VitsTokenizer.from_pretrained(
        str(local),
        local_files_only=True
    )

    model = VitsModel.from_pretrained(
        str(local),
        local_files_only=True,
        use_safetensors=False
    ).to(DEVICE)

    model.eval()
    sr = model.config.sampling_rate

    for i, sentence in enumerate(SENTENCES[lang], 1):
        path = output / f"test_{i:02d}.wav"

        if path.exists():
            continue

        inputs = tokenizer(sentence, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            audio = model(**inputs).waveform[0].cpu().numpy()

        audio = np.nan_to_num(audio).astype(np.float32)
        sf.write(path, audio, sr)

        print(f"{lang}: {i}/5 saved")

    display(Audio(filename=str(output / "test_02.wav")))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
from pathlib import Path
from IPython import get_ipython

OUT = Path("/kaggle/working/Recovered_Notebook_Cells.py")

ip = get_ipython()

if ip is None:
    print("IPython session unavailable")
else:
    history = ip.history_manager.get_range(
        session=0,
        start=1,
        stop=None,
        raw=True,
        output=False
    )

    cells = []

    for session, line, source in history:
        if not source.strip():
            continue

        cells.append(
            f"\n\n# ===== CELL {line} =====\n\n{source}"
        )

    OUT.write_text(
        "".join(cells),
        encoding="utf-8"
    )

    print("Recovered cells:", len(cells))
    print("Saved to:", OUT)

In [ ]:
from pathlib import Path
import zipfile

ROOT = Path("/kaggle/working/FastSpeech2_Submission")
OUT = Path("/kaggle/working/FastSpeech2_Recovery.zip")

exclude_dirs = {
    ".git",
    "wavs",
    "acoustic_features",
    "alignment_test",
    "pretrained_models",
    "__pycache__"
}

with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        if not p.is_file():
            continue

        rel = p.relative_to(ROOT)

        if any(part in exclude_dirs for part in rel.parts):
            continue

        z.write(p, rel)

    recovered = Path("/kaggle/working/Recovered_Notebook_Cells.py")

    if recovered.exists():
        z.write(recovered, recovered.name)

print("Created:", OUT)
print("Size:", round(OUT.stat().st_size / 1024**2, 1), "MB")

In [ ]:
from pathlib import Path

WORK = Path("/kaggle/working")

candidates = [
    WORK / "FastSpeech2_Recovery.zip",
    WORK / "Final_Submission.zip",
    WORK / "FastSpeech2_Submission_Backup.zip"
]

for p in candidates:
    if p.exists():
        print(f"{p.name}: {p.stat().st_size / 1024**2:.1f} MB")

In [ ]:
from pathlib import Path

WORK = Path("/kaggle/working")
SOURCE = WORK / "FastSpeech2_Recovery.zip"

assert SOURCE.exists(), "Recovery ZIP not found"

PART_SIZE = 100 * 1024 * 1024

with SOURCE.open("rb") as src:
    i = 1

    while True:
        data = src.read(PART_SIZE)

        if not data:
            break

        target = WORK / f"recovery_part_{i:03d}.part"
        target.write_bytes(data)

        print(f"{target.name}: {len(data) / 1024**2:.1f} MB")
        i += 1

print("Total parts:", i - 1)

In [ ]:
from IPython.display import display, FileLink
from pathlib import Path

for p in sorted(Path("/kaggle/working").glob("recovery_part_*.part")):
    display(FileLink(str(p)))